# Chapter 14 Laboratory

Turn a linear score into a probability and train a binary classifier from [`blog.md`](<blog.md>).

## Prediction
Before running the code, predict the sigmoid values at `z=-2`, `0`, and `2`. Then predict which side of `z=0` should correspond to class 1.

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

for z in [-2, 0, 2]:
    print(z, sigmoid(z))


## Mathematics

`z = X @ weights + bias`, `p = sigmoid(z)`, and binary cross-entropy is the negative Bernoulli log-likelihood.

In [ ]:
x_one = np.array([2., 1.])
w_one = np.array([1., -0.5])
b_one = -0.5
y_one = 1.
z_one = x_one @ w_one + b_one
p_one = sigmoid(z_one)
grad_scale = p_one - y_one
assert np.isclose(z_one, 1.0)
assert np.isclose(p_one, 0.7310585786300049)
assert np.isclose(grad_scale, -0.2689414213699951)


In [ ]:
X = np.array([[0.,1.], [1.,1.], [3.,2.], [4.,3.]])
y = np.array([0.,0.,1.,1.])
weights = np.zeros(2)
bias = 0.0
learning_rate = 0.2
history = []

for step in range(3000):
    z = X @ weights + bias
    p = sigmoid(z)
    eps = 1e-12
    loss = -np.mean(y*np.log(np.clip(p, eps, 1-eps)) + (1-y)*np.log(np.clip(1-p, eps, 1-eps)))
    error = p - y
    grad_w = (X.T @ error) / len(X)
    grad_b = error.mean()
    weights -= learning_rate * grad_w
    bias -= learning_rate * grad_b
    history.append(loss)

print(weights, bias, history[-1])
assert history[-1] < history[0]


In [ ]:
import matplotlib.pyplot as plt

plt.plot(history)
plt.xlabel('step')
plt.ylabel('binary cross-entropy')
plt.title('Logistic regression training loss')
plt.show()


In [ ]:
grid_x1 = np.linspace(-1, 5, 200)
grid_x2 = np.linspace(0, 4, 200)
xx1, xx2 = np.meshgrid(grid_x1, grid_x2)
grid = np.column_stack([xx1.ravel(), xx2.ravel()])
probs = sigmoid(grid @ weights + bias).reshape(xx1.shape)

plt.contourf(xx1, xx2, probs, levels=[0, .5, 1], alpha=0.25)
plt.scatter(X[:,0], X[:,1], c=y)
plt.xlabel('suspicious words')
plt.ylabel('links')
plt.title('Probability surface and decision boundary')
plt.show()


In [ ]:
# Move one positive example farther from the boundary.
X_changed = X.copy()
X_changed[2] = [5., 3.]

def train(X, y, steps=3000, lr=0.2):
    w = np.zeros(X.shape[1]); b = 0.0
    for _ in range(steps):
        p = sigmoid(X @ w + b)
        e = p-y
        w -= lr * (X.T @ e)/len(X)
        b -= lr * e.mean()
    return w, b

w_changed, b_changed = train(X_changed, y)
print('original:', weights, bias)
print('changed :', w_changed, b_changed)


## Observe
The loss should fall during training. The sigmoid outputs probabilities, and the 0.5 contour corresponds to the linear score `z=0`.

## Explain
The Bernoulli negative log-likelihood produces a simple gradient: the prediction error `p-y` multiplied by each feature. The sigmoid and log-loss derivatives cancel in the chain rule.

In [ ]:
# Challenge: try a threshold of 0.8 instead of 0.5 and compare the predicted classes.
# Then experiment with a much larger learning rate and inspect the loss curve.
# YOUR CODE HERE


## Reflection
- I can explain why a raw linear score is not a probability.
- I can derive the sigmoid/log-odds relationship.
- I understand why binary cross-entropy comes from Bernoulli likelihood.
- I can derive the `(p-y)x` gradient.

Next: generalize one binary probability to many classes.